In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/project1/setup_catalogs/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog', 'project1_cat', 'catalog')
dbutils.widgets.text('data_source', 'orders', 'data_source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sportsbar-childpro/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'
print('base_path:', base_path)
print('landing_path:', landing_path)
print('processed_path:', processed_path)

#Define the table
bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.sb_fact_{data_source}'

In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_timestamp', F.current_timestamp())

In [0]:
print('Total rows: ', df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
df.write.format('delta').option('delta.enableChangeDataFeed', 'true').mode('append').saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(file_info.path, f'{processed_path}/{file_info.name}',True)

In [0]:
df_orders = spark.sql(f'select * from {bronze_table}')

display(df_orders.limit(20))

### Silver

Transformation

In [0]:
# keep only rows where the quantity is present
df_orders = df_orders.filter(F.col('order_qty').isNotNull())

# clean customer_id -> keep numeric, else set to 99999
df_orders = df_orders.withColumn('customer_id', F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id')).otherwise(99999).cast('string'))

#Remove weekday name from date text 'Tuesday, July 01, 2025' -> 'July 01, 2025'
df_orders = df_orders.withColumn('order_placement_date', 
                                 F.regexp_replace(F.col('order_placement_date'), r"^[A-Za-z]+,\s*",""))

#parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn('order_placement_date', 
                                 F.coalesce(
                                     F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
                                     F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
                                     F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
                                     F.try_to_date('order_placement_date', 'MMMM dd, yyyy')
                                 ))

#Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# Convert product_id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
#Check what is minimum and maximum date

df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).show()

In [0]:
display(df_orders.limit(50))

In [0]:
df_products = spark.table('project1_cat.silver.products')

display(df_products.limit(20))

In [0]:
df_joined = df_orders.join(df_products, on='product_id', how='inner').select(df_orders['*'], df_products['product_code'])

display(df_joined)

In [0]:
if not (spark.catalog.tableExists('project1_cat.silver.orders')):
    df_joined.write.format('delta').option('delta.enableChangeDataFeed', 'true').option('mergeSchema', 'true').mode('overwrite').saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    